In [4]:
!pip install unsloth
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

  Using cached unsloth-2026.3.4-py3-none-any.whl.metadata (70 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached unsloth-2026.3.4-py3-none-any.whl (447 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.2 MB/s eta 0:00:0

In [5]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [7]:
prompt = "Explain the mechanism of action for Imatinib in treating CML."

inputs = tokenizer(
    [f"### Question: {prompt}\n### Answer: "],
    return_tensors="pt"
).to("cuda")

print("\n--- BASELINE GENERATION ---")

outputs = model.generate(
    **inputs,
    max_new_tokens=64
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


--- BASELINE GENERATION ---


--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

### Question: Explain the mechanism of action for Imatinib in treating CML.
### Answer: 1. It is a tyrosine kinase inhibitor.
    - It inhibits the BCR-ABL tyrosine kinase and also has some activity against the C-KIT and PDGF receptors.
    - It is a small molecule, which binds to the active site of the BCR-ABL tyrosine kinase


In [10]:
def format_prompts(examples):
    texts = [
        f"### Question: {q}\n### Answer: {a}"
        for q, a in zip(examples['instruction'], examples['output'])
    ]
    return { "text" : texts }

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = dataset.map(format_prompts, batched=True)

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [12]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer, # Add the tokenizer here
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 40,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/51760 [00:00<?, ? examples/s]

In [13]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,1.514791
2,1.787116
3,1.808166
4,1.792907
5,1.350475
6,1.291196
7,1.291512
8,1.419192
9,1.268610
10,1.451841


TrainOutput(global_step=40, training_loss=1.2492556259036065, metrics={'train_runtime': 204.8697, 'train_samples_per_second': 1.562, 'train_steps_per_second': 0.195, 'total_flos': 3211252586397696.0, 'train_loss': 1.2492556259036065, 'epoch': 0.0061823802163833074})

In [14]:
print("\n--- POST-TUNING GENERATION ---")

outputs = model.generate(
    **inputs,
    max_new_tokens=64
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


--- POST-TUNING GENERATION ---
### Question: Explain the mechanism of action for Imatinib in treating CML.
### Answer: 1. Imatinib is a small molecule tyrosine kinase inhibitor that targets the BCR-ABL fusion protein, which is formed by the translocation of the Abelson tyrosine kinase gene (ABL) and the breakpoint cluster region (BCR) gene in chronic myeloid leukemia (CML). 




Imatinib functions as a tyrosine kinase inhibitor which specifically targets the BCR-ABL fusion protein that results from the Philadelphia chromosome in chronic myeloid leukemia (CML) to block ATP binding and stop the growth of leukemic cells.

Why did the model's vocabulary get more technical? Did we give it a textbook, or just 200 conversations?

The model did not receive a full medical textbook. Instead, it was fine-tuned using 200 medical question-and-answer conversations. The conversations included medical terms which doctors use and clinical thinking methods. The model used LoRA adaptation to change its internal structure by using attention and MLP layers to improve its capacity for understanding specialized language and domain-specific knowledge. The model generated responses which used the medical professionals' style and vocabulary. The base model's general knowledge enables the model to produce specialized outputs from a small dataset while LoRA fine-tuning directs it to the specific domain.